# PDF Q&A with RAG (Nike 10-K)
Run with **Runtime → Restart and run all**. Needs Colab secrets `GEMINI_API_KEY` (and optionally `LANG_SMITH_KEY`, `HF_TOKEN`).

In [ ]:
!pip install -qU pypdf langchain-core langchain-google-genai langchain-text-splitters langchain-huggingface

## 1. Setup: API keys and Google Drive

In [ ]:
import os
from google.colab import userdata, drive

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# Optional: LangSmith tracing (lets you see which chunks were retrieved)
try:
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANG_SMITH_KEY")
    os.environ["LANGSMITH_TRACING"] = "true"
except Exception:
    print("LangSmith key not found - tracing off")

# Optional: removes the HF Hub rate-limit warning
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

# Drive keeps the vector store between sessions (plain /content is wiped on reset)
drive.mount("/content/drive")
STORE_PATH = "/content/drive/MyDrive/nike_store.json"

## 2. Load the PDF

In [ ]:
import pypdf
from langchain_core.documents import Document

PDF_PATH = "/content/nke-10k-2023.pdf"  # upload it via the Files panel, or point to a Drive path

def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(page_content=page.extract_text() or "",
                 metadata={"source": file_path, "page": i})
        for i, page in enumerate(reader.pages)
    ]

docs = load_pdf_pages(PDF_PATH)
print(f"Loaded {len(docs)} pages")

## 3. Split into chunks
800 characters fits inside the embedding model's ~256-token limit, so nothing gets cut off.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=150, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)
print(f"Created {len(all_splits)} chunks")

## 4. Embeddings and vector store
Loads the saved store from Drive if it exists; otherwise builds and saves it. Set `REBUILD = True` after changing the PDF, chunk settings or embedding model.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

REBUILD = False

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if os.path.exists(STORE_PATH) and not REBUILD:
    vector_store = InMemoryVectorStore.load(STORE_PATH, embeddings)
    print("Loaded saved vector store from Drive")
else:
    vector_store = InMemoryVectorStore(embeddings)
    vector_store.add_documents(documents=all_splits)
    vector_store.dump(STORE_PATH)
    print(f"Built and saved vector store ({len(all_splits)} chunks)")

## 5. LLM, prompt and retriever

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

# MMR = relevant AND varied chunks, so broad questions get wider coverage
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 6})

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions about a company report using only the context below. "
     "Each chunk starts with its page number; cite pages like (p. 12). "
     "If the context does not contain the answer, say you don't know. "
     "Keep the answer to three or four sentences.\n\n{context}"),
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

## 6. Ask questions

In [ ]:
def ask(question: str) -> None:
    retrieved = retriever.invoke(question)
    context = "\n\n".join(
        f"[Page {d.metadata['page'] + 1}]\n{d.page_content}" for d in retrieved
    )
    print("Q:", question)
    print(chain.invoke({"context": context, "question": question}))
    print("Sources: pages", sorted({d.metadata["page"] + 1 for d in retrieved}))
    print("-" * 80)

ask("What were some of the main business risks mentioned in the report?")

## 7. Quick test set
Check these answers against the PDF. The last question isn't in the report, so the model should say it doesn't know.

In [ ]:
test_questions = [
    "What was Nike's total revenue in fiscal 2023?",
    "How many employees did Nike have?",
    "Where are most Nike products manufactured?",
    "What is Nike's strategy for direct-to-consumer sales?",
    "Who won the 2022 FIFA World Cup?",  # not in the report
]
for q in test_questions:
    ask(q)